In [ ]:
from pathlib import Path

import anndata as ad
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.ndimage import binary_dilation, gaussian_filter
from scipy.spatial import cKDTree
from skimage.segmentation import find_boundaries, mark_boundaries
 
DATA_FILE = "../../Pt7-3a_data.h5ad"
OUT_DIR =   Path("../../")

adata = sc.read_h5ad(DATA_FILE)
adata

In [ ]:
final_mask = np.squeeze(np.asarray(adata.uns["final_mask"])).astype(bool)
img = np.asarray(adata.uns["staining_image"])
 
if img.shape[:2] != final_mask.shape:
    if img.shape[:2][::-1] == final_mask.shape:
        img = img.transpose(1, 0, 2) if img.ndim == 3 else img.T
if img.ndim == 2:
    img = np.stack([img] * 3, axis=-1)

if img.shape[2] > 3:
    img = img[:, :, :3]
if img.dtype != np.uint8:
    if np.nanmax(img) <= 1:
        img = img * 255
    img = np.clip(img, 0, 255).astype(np.uint8)
H, W = img.shape[:2]
img_float = img.astype(np.float32) / 255.0
alpha = final_mask.astype(np.float32)
img_rgba = np.concatenate([img_float, alpha[..., None]], axis=-1)
print("Figure-D canvas (H, W):", (H, W))


In [ ]:
coords = np.asarray(adata.obsm["spatial"])
x, y = coords[:, 0], coords[:, 1]
ncount = pd.to_numeric(adata.obs["total_counts"], errors="coerce").to_numpy(dtype=float)
 
grid_res = 1000
density = np.zeros((grid_res, grid_res), dtype=np.float64)
weights = (ncount - ncount.min()) / (ncount.max() - ncount.min() + 1e-8)
x_idx = ((x / W) * (grid_res - 1)).astype(int).clip(0, grid_res - 1)
y_idx = ((y / H) * (grid_res - 1)).astype(int).clip(0, grid_res - 1)
for xi, yi, weight in zip(x_idx, y_idx, weights):
    density[yi, xi] += weight

density_smooth = gaussian_filter(density, sigma=3)
positive = density_smooth[density_smooth > 0]
 
signal_vmax = np.nanpercentile(positive, 99)
signal_norm = np.clip(density_smooth / (signal_vmax + 1e-8), 0, 1)
display_norm = signal_norm ** 0.7


In [ ]:
hex_data = adata.uns["hex_data"]
hex_xy = np.asarray(hex_data["spatial"], dtype=float)
hex_genes = [str(g) for g in np.asarray(hex_data["genes"]).tolist()]
hex_values = np.asarray(hex_data["raw_marker_values"], dtype=float)
 
tree = cKDTree(hex_xy)
dist, idx = tree.query(np.asarray(adata.obsm["spatial"][:, :2], dtype=float), k=1)
adata.obs["istar15_hex_id"] = np.asarray(hex_data["obs_names"])[idx].astype(str)
adata.obs["istar15_dist"] = dist
adata.obs["istar15_mapped"] = True  # source max_dist=None


def normalize_for_spatial_plot(values, truncate=(-3, 3)):
    values = np.asarray(values, dtype=float).copy()
    if truncate is not None:
        values -= np.nanmean(values)
        values /= np.nanstd(values) + 1e-12
        values = np.clip(values, truncate[0], truncate[1])
    values -= np.nanmin(values)
    values /= np.nanmax(values) + 1e-12
    return values


for j, gene in enumerate(hex_genes):
    raw_cell = hex_values[idx, j]
    adata.obs[f"istar15_{gene}_pred"] = raw_cell
    adata.obs[f"{gene}_norm"] = normalize_for_spatial_plot(raw_cell, truncate=(-3, 3))

print(pd.Series(dist).describe())


In [ ]:
CT_colors = {
    "Bcell": "#ba55d3", "Endothelial": "#1E90FF", "Epithelial": "#9fa6ab",
    "Fibroblast": "#BD8F8F", "Myeloid": "#FFD500", "Hepatocyte": "#639d63",
    "Tcell": "#FF0000", "Unknown": "#E3E1E1",
}

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(img_rgba)
ax.axis("off")
fig.savefig(OUT_DIR / "HE.png", dpi=300, bbox_inches="tight", pad_inches=0, facecolor="white")
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 6), dpi=150)
ax.imshow(
    display_norm, origin="upper", cmap="viridis", extent=[0, W, H, 0],
    vmin=0, vmax=1.0, interpolation="bilinear",
)
ax.set_xlim(0, W)
ax.set_ylim(H, 0)
ax.axis("off")
fig.savefig(OUT_DIR / "total_count.pdf", dpi=500, bbox_inches="tight", pad_inches=0, facecolor="white")
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 6))
sc.pl.spatial(
    adata, img=img_rgba, alpha_img=0.1, color="CellType", basis="spatial",
    spot_size=35, palette=CT_colors, legend_loc=None, frameon=False, ax=ax, show=False,
)
ax.set_xlim(0, W)
ax.set_ylim(H, 0)
ax.axis("off")
fig.savefig(OUT_DIR / "celltype_R1.pdf", bbox_inches="tight", dpi=300)
plt.close(fig)


## Plot Figure E — whole-section marker panels

Marker set, cell-type restrictions, H&E opacity, point size, percentile cap, and colormap come from source Cells 33, 40, and 41.


In [ ]:
E_GENES = [
    "EPCAM", "ALB", "COL1A2", "ACTA2", "ENG", "VWF", "PECAM1",
    "CDH5", "PTPRC", "CD68", "C1QA", "CD3E", "CD3D", "CD79A",
]
gene_celltype_map = {
    "EPCAM": ["Epithelial"], "ALB": ["Hepatocyte"],
    "COL1A2": ["Fibroblast"], "ACTA2": ["Fibroblast"],
    "ENG": ["Endothelial"], "PECAM1": ["Endothelial"],
    "VWF": ["Endothelial"], "CDH5": ["Endothelial"],
    "PTPRC": ["Tcell", "Bcell", "Myeloid"], "CD68": ["Myeloid"],
    "C1QA": ["Myeloid"], "CD3E": ["Tcell"], "CD3D": ["Tcell"],
    "CD79A": ["Bcell"],
}

fig, axes = plt.subplots(1, len(E_GENES), figsize=(5.2 * len(E_GENES), 5.2), dpi=300)
main_axes = list(np.ravel(axes))
for ax, gene in zip(main_axes, E_GENES):
    valid_cts = [ct for ct in gene_celltype_map[gene] if ct in adata.obs["CellType"].unique()]
    if not valid_cts:
        ax.set_title(f"{gene}\nno target cells")
        ax.axis("off")
        continue
    adata_sub = adata[adata.obs["CellType"].isin(valid_cts)].copy()
    sc.pl.spatial(
        adata_sub, basis="spatial", img=img_rgba, alpha_img=0.1,
        color=f"{gene}_norm", spot_size=80, vmax="p99", cmap="YlOrRd",
        ax=ax, show=False, title=gene,
    )
    ax.set_xlim(0, W)
    ax.set_ylim(H, 0)
    ax.set_aspect("equal")
    ax.axis("off")
    for other_ax in list(fig.axes):
        if other_ax not in main_axes:
            other_ax.remove()

plt.tight_layout(w_pad=0.1)
fig.savefig(OUT_DIR / "AllMarker_targetCellTypeOnly_marker.pdf", bbox_inches="tight", dpi=300)
plt.close(fig)

In [ ]:
spot_data = adata.uns["spot_data"]
spot_cell_id = pd.to_numeric(pd.Series(np.asarray(spot_data["cell_id"])), errors="coerce").astype("Int64")
spot_celltype = pd.Series(np.asarray(spot_data["CellType_ficture"]).astype(str))
spot_xy = np.asarray(spot_data["spatial_manual_fine"], dtype=float)
if not (len(spot_cell_id) == len(spot_celltype) == len(spot_xy)):
    raise ValueError("spot arrays have inconsistent lengths")

seg_img_original = np.asarray(adata.uns["staining_image"])
seg_mask_original = np.asarray(adata.uns["staining_image_mask"])
spatial_hw = tuple(final_mask.shape)


def crop_array_by_spatial_bbox(arr, xmin, ymin, xmax, ymax, spatial_hw, key="array"):
    arr = np.asarray(arr)
    hw = arr.shape[:2]
    if hw == tuple(spatial_hw):
        return arr[ymin:ymax, xmin:xmax, ...]
    if hw == tuple(spatial_hw)[::-1]:
        return arr[xmin:xmax, ymin:ymax, ...]
    raise ValueError(f"{key}: shape mismatch {hw}, spatial={spatial_hw}")


def prepare_segmentation_crop(bbox, method):
    xmin, ymin, xmax, ymax = bbox
    # These two source arrays are in reversed image orientation.
    img_crop = crop_array_by_spatial_bbox(
        seg_img_original, xmin, ymin, xmax, ymax, spatial_hw, "staining_image"
    )
    mask_crop = crop_array_by_spatial_bbox(
        seg_mask_original, xmin, ymin, xmax, ymax, spatial_hw, "staining_image_mask"
    )

    labels, counts = np.unique(mask_crop, return_counts=True)
    keep_labels = labels[(labels != 0) & (counts >= 10)]  # source min_pixels=10
    target_mask = mask_crop.copy()
    target_mask[~np.isin(target_mask, keep_labels)] = 0

    img_show = img_crop.transpose(1, 0, 2) if img_crop.ndim == 3 else img_crop.T
    target_mask = target_mask.T

    if method == "find":
        boundary = find_boundaries(target_mask.astype(int), mode="inner")
    elif method == "mark":
        blank_rgb = np.zeros((*target_mask.shape, 3), dtype=np.float32)
        marked = mark_boundaries(
            blank_rgb, target_mask.astype(int), color=(0, 1, 0),
            mode="thick", background_label=0,
        )
        boundary = binary_dilation(marked[:, :, 1] > 0, iterations=1)
    else:
        raise ValueError(method)
    return {"img": img_show, "label_mask": target_mask, "boundary": boundary}


def boundary_rgba(boundary, alpha=1.0):
    layer = np.zeros((*boundary.shape, 4), dtype=np.float32)
    layer[boundary, 1] = 1.0
    layer[boundary, 3] = alpha
    return layer


cell_ids = pd.to_numeric(adata.obs["cell_ID_mask"], errors="coerce").astype("Int64")


def prepare_spot_region(bbox, genes, gene_to_celltype, celltype_patterns):
    xmin, ymin, xmax, ymax = bbox
    inside = (
        (spot_xy[:, 0] >= xmin) & (spot_xy[:, 0] < xmax)
        & (spot_xy[:, 1] >= ymin) & (spot_xy[:, 1] < ymax)
    )
    if inside.sum() == 0:
        raise ValueError(f"No spots in bbox {bbox}")

    local_xy = spot_xy[inside] - np.array([xmin, ymin])
    local_xy = local_xy[:, [1, 0]]  # source Cell 47 coordinate swap
    local_ids = spot_cell_id[inside].reset_index(drop=True)
    local_ct = spot_celltype[inside].reset_index(drop=True)

    img_raw = crop_array_by_spatial_bbox(
        seg_img_original, xmin, ymin, xmax, ymax, spatial_hw, "staining_image"
    )
    mask_raw = crop_array_by_spatial_bbox(
        seg_mask_original, xmin, ymin, xmax, ymax, spatial_hw, "staining_image_mask"
    )
    img_show = img_raw.transpose(1, 0, 2) if img_raw.ndim == 3 else img_raw.T
    mask_show = np.squeeze(mask_raw).T
    if mask_show.shape != img_show.shape[:2]:
        raise ValueError(f"Window image/mask mismatch: {img_show.shape[:2]} vs {mask_show.shape}")
    marked = mark_boundaries(np.zeros_like(img_show), mask_show, color=(0, 1, 0), mode="thick")
    spot_boundary = marked[:, :, 1] > 0

    obs = pd.DataFrame({"CellType_ficture": local_ct.to_numpy(), "cell_id": local_ids.to_numpy()})
    for gene in genes:
        expr_map = pd.DataFrame({
            "cell_id": cell_ids.to_numpy(),
            "value": pd.to_numeric(adata.obs[f"{gene}_norm"], errors="coerce").to_numpy(),
        }).dropna(subset=["cell_id"]).drop_duplicates("cell_id").set_index("cell_id")["value"]
        values = pd.to_numeric(local_ids.map(expr_map), errors="coerce")
        pattern = celltype_patterns[gene_to_celltype[gene]]
        valid_ct = local_ct.str.contains(pattern, case=False, na=False, regex=True)
        values.loc[~valid_ct] = np.nan
        obs[f"{gene}_plot"] = values.to_numpy(dtype=np.float32)
        print(gene, "valid spots:", values.notna().sum())

    return {"xy": local_xy.astype(np.float32), "obs": obs, "img": img_show, "boundary": spot_boundary}


def region_as_anndata(region):
    out = ad.AnnData(X=np.zeros((len(region["obs"]), 0), dtype=np.float32), obs=region["obs"].copy())
    out.obsm["spatial_manual_fine"] = region["xy"]
    out.obsm["spatial"] = region["xy"]
    return out


In [ ]:
F_BBOX = (2650, 7100, 3150, 7600)
F_GENES = ["PTPRC", "CD68", "CD3E", "CD3D", "CD79A"]
F_GENE_TO_CELLTYPE = {
    "PTPRC": "Immune", "CD68": "Myeloid", "CD3E": "Tcell",
    "CD3D": "Tcell", "CD79A": "Bcell",
}
F_PATTERNS = {
    "Immune": "Myeloid|Tcell|Bcell|Immune|Macrophage|Mono|Lymph",
    "Myeloid": "Myeloid|Macrophage|Mono|Monocyte",
    "Tcell": "Tcell|T_cell|T cell|CD4|CD8",
    "Bcell": "Bcell|B_cell|B cell|Plasma",
}

F_seg = prepare_segmentation_crop(F_BBOX, method="find")
F = prepare_spot_region(F_BBOX, F_GENES, F_GENE_TO_CELLTYPE, F_PATTERNS)
F_boundary = boundary_rgba(F_seg["boundary"], alpha=1.0)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(F_seg["img"])
ax.imshow(F_boundary)
ax.axis("off")
fig.savefig(OUT_DIR / "Subset_immune.pdf", bbox_inches="tight", dpi=300)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(F["img"], alpha=0.3, origin="upper")
for name, color in {"Myeloid": "#F6E600", "Tcell": "#F60000", "Bcell": "#C900F6"}.items():
    target = F["obs"]["CellType_ficture"].to_numpy() == name
    ax.scatter(F["xy"][target, 0], F["xy"][target, 1], s=8, c=color,
               alpha=1, linewidths=0, zorder=2, rasterized=True, label=name)
ax.imshow(boundary_rgba(F["boundary"], alpha=0.65), origin="upper", zorder=3)
ax.set_xlim(0, F["img"].shape[1])
ax.set_ylim(F["img"].shape[0], 0)
ax.set_aspect("equal")
ax.axis("off")
fig.savefig(OUT_DIR / "Immune_subtypes_combined.pdf", bbox_inches="tight", dpi=300)
plt.close(fig)

F_adata = region_as_anndata(F)
fig, axes = plt.subplots(1, len(F_GENES), figsize=(6 * len(F_GENES), 6))
main_axes = list(np.ravel(axes))
for ax, gene in zip(main_axes, F_GENES):
    plot_col = f"{gene}_plot"
    sub = F_adata[F_adata.obs[plot_col].notna()].copy()
    if sub.n_obs:
        sc.pl.spatial(sub, img=F["img"], alpha_img=0.35, basis="spatial_manual_fine",
                      color=plot_col, spot_size=8, ax=ax, show=False, title=gene,
                      cmap="YlOrRd", vmax="p99", frameon=False)
    else:
        ax.imshow(F["img"], alpha=0.35, origin="upper")
        ax.set_title(f"{gene}\nNo spots")
    ax.imshow(boundary_rgba(F["boundary"], alpha=1.0), origin="upper", zorder=4)
    ax.set_xlim(0, F["img"].shape[1])
    ax.set_ylim(F["img"].shape[0], 0)
    ax.axis("off")
    for other_ax in list(fig.axes):
        if other_ax not in main_axes:
            other_ax.remove()
plt.tight_layout()
fig.savefig(OUT_DIR / "Immune_marker_win.pdf", bbox_inches="tight", dpi=300)
plt.close(fig)


In [ ]:
G_BBOX = (2700, 9200, 3450, 9900)
G_GENES = ["PECAM1", "VWF", "COL1A2", "ACTA2"]
G_GENE_TO_CELLTYPE = {
    "PECAM1": "Endothelial", "VWF": "Endothelial",
    "COL1A2": "Fibroblast", "ACTA2": "Fibroblast",
}
G_PATTERNS = {
    "Endothelial": "Endothelial|Endothe",
    "Fibroblast": "Fibroblast|Fibro|CAF|Stromal",
}

G_seg = prepare_segmentation_crop(G_BBOX, method="mark")
G = prepare_spot_region(G_BBOX, G_GENES, G_GENE_TO_CELLTYPE, G_PATTERNS)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(G_seg["img"])
ax.imshow(boundary_rgba(G_seg["boundary"], alpha=1.0))
ax.axis("off")
fig.savefig(OUT_DIR / "G_subset_endothelial.pdf", bbox_inches="tight", dpi=300)
plt.show()
plt.close(fig)

for celltype_name, out_name, color in [
    ("Endothelial", "G_endothelial_spots.pdf", "#1E90FF"),
    ("Fibroblast", "G_fibroblast_spots.pdf", "red"),
]:
    target = G["obs"]["CellType_ficture"].to_numpy() == celltype_name
    if not target.any():
        print(f"No {celltype_name} spots; skipped")
        continue
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(G["img"], alpha=0.9, origin="upper")
    ax.scatter(G["xy"][target, 0], G["xy"][target, 1], s=3, c=color,
               alpha=0.95, linewidths=0, zorder=2, rasterized=True)
    ax.imshow(boundary_rgba(G["boundary"], alpha=1.0), origin="upper", zorder=3)
    ax.set_xlim(0, G["img"].shape[1])
    ax.set_ylim(G["img"].shape[0], 0)
    ax.set_aspect("equal")
    ax.axis("off")
    fig.savefig(OUT_DIR / out_name, bbox_inches="tight", dpi=300)
    plt.show()
    plt.close(fig)

G_adata = region_as_anndata(G)
fig, axes = plt.subplots(1, len(G_GENES), figsize=(6 * len(G_GENES), 6))
main_axes = list(np.ravel(axes))
for ax, gene in zip(main_axes, G_GENES):
    plot_col = f"{gene}_plot"
    sub = G_adata[G_adata.obs[plot_col].notna()].copy()
    if sub.n_obs:
        sc.pl.spatial(sub, img=G["img"], alpha_img=0.35, basis="spatial_manual_fine",
                      color=plot_col, spot_size=5, ax=ax, show=False, title=gene,
                      cmap="YlOrRd", vmax="p99", frameon=False)
    else:
        ax.imshow(G["img"], alpha=0.35, origin="upper")
        ax.set_title(f"{gene}\nNo spots")
    ax.imshow(boundary_rgba(G["boundary"], alpha=1.0), origin="upper", zorder=4)
    ax.set_xlim(0, G["img"].shape[1])
    ax.set_ylim(G["img"].shape[0], 0)
    ax.axis("off")
    for other_ax in list(fig.axes):
        if other_ax not in main_axes:
            other_ax.remove()
plt.tight_layout()
fig.savefig(OUT_DIR / "G_Endothelial_Fibroblast_marker_win.pdf", bbox_inches="tight", dpi=300)
plt.show()
plt.close(fig)
